# Volume 13, Chapter 14: The critical line (Python companion)

Exercises first, solutions at the end. We check Hardy's identity and watch the moments of $\Xi$ against $\cosh$ stay bounded, test Hilbert's inequality, measure the mean square of $\zeta$ on the critical line, verify Montgomery's explicit formula with real zeros and primes, and compute his pair correlation function. The computer science application: **the fast evaluation of $\zeta$**, with the Riemann–Siegel formula and the multi-evaluation idea of Odlyzko and Schönhage, used to find every zero up to height $10^4$. Requires `numpy` and `mpmath`.

In [ ]:
import math, time
import numpy as np
import mpmath as mp
mp.mp.dps = 20

def theta_rs(t):
    # Riemann-Siegel theta function, Stirling series (Volume 5, equation (6.4))
    return t/2*np.log(t/(2*np.pi)) - t/2 - np.pi/8 + 1/(48*t) + 7/(5760*t**3)

def Z_rs(t):
    # Hardy's Z(t) by the Riemann-Siegel formula with the first correction term (Theorem 14.11)
    t = np.atleast_1d(np.asarray(t, float))
    a = np.sqrt(t/(2*np.pi)); N = np.floor(a).astype(int); p = a - N
    out = np.zeros_like(t); th = theta_rs(t)
    for n in range(1, N.max()+1):
        m = N >= n
        out[m] += 2*np.cos(th[m] - t[m]*math.log(n))/math.sqrt(n)
    C0 = np.cos(2*np.pi*(p*p - p - 1/16))/np.cos(2*np.pi*p)
    return out + (-1.0)**(N-1)*a**-0.5*C0

def zeros_rs(T0, T1, step=0.01):
    # all sign changes of Z on [T0,T1], refined by vectorized bisection
    t = np.arange(T0, T1, step); z = Z_rs(t)
    i = np.where(np.sign(z[:-1]) != np.sign(z[1:]))[0]
    a, b, fa = t[i], t[i+1], z[i]
    for _ in range(30):
        c = (a+b)/2; fc = Z_rs(c); same = np.sign(fc) == np.sign(fa)
        a = np.where(same, c, a); fa = np.where(same, fc, fa); b = np.where(same, b, c)
    return (a+b)/2

## Exercises

**P14.1 (★) Hardy's identity.** Check equation (14.1) at $x=0,\frac12,1$. Then compute $\Phi(\alpha)=\int_0^\infty\frac{\Xi(t)}{t^2+1/4}\cosh(\alpha t)\,dt$ from the closed form $\frac\pi2\bigl(2\cos\frac\alpha2-e^{i\alpha/2}\theta(e^{2i\alpha})\bigr)$ for $\alpha\to\frac\pi4$, using Lemma 14.2 to evaluate $\theta$ near $i$, and compare with the limit $\pi\cos\frac\pi8$. Finally look at the size of the integrand $\frac{\Xi(t)}{t^2+1/4}\cosh\frac{\pi t}4$: how can $\Phi$ have a limit?

**P14.2 (★) Hilbert's inequality.** Compute the largest modulus of an eigenvalue of the $N\times N$ matrix $\bigl(\frac1{m-n}\bigr)_{m\ne n}$ for $N=50,100,\dots,1600$ and compare with $\pi$. Then check Proposition 14.8: compute $\int_0^T|S(t)|^2\,dt-T\sum_{n\le N}\frac1n$ for $S(t)=\sum_{n\le N}n^{-1/2-it}$ and several $N$, $T$.

**P14.3 (★★) The mean square.** Check Lemma 14.6 at random points $\frac12+it$ with $x=t+3$: how large is $|E|x^{1/2}$? Then compute $\frac1T\int_0^T|\zeta(\frac12+it)|^2dt-\log\frac T{2\pi}$ for $T$ up to $2000$ and compare with Ingham's $2\gamma-1$.

**P14.4 (★★) Montgomery's explicit formula and $F(\alpha)$.** With the zeros up to $10^4$ and the primes up to $10^7$, check Lemma 14.13 at $(x,t)=(10,100)$, $(50,30)$, $(200,500)$. Then compute Montgomery's $F(\alpha)$ for $T=5000$ (with the zeros unfolded to mean spacing 1) and compare with $\alpha+\ell e^{-2\alpha\ell}$, $\ell=\log\frac T{2\pi}$, for $\alpha<1$ and with $1$ beyond.

**P14.5 (★★, computer science) Evaluating $\zeta$ fast.** (a) Compare the cost and accuracy of Lemma 14.6 (about $t$ terms), the Riemann–Siegel formula ($\sqrt{t/2\pi}$ terms) and `mpmath`. (b) Find all zeros up to $10^4$ and check the count against the Riemann–von Mangoldt formula $N(T)=\frac{\vartheta(T)}\pi+1+S(T)$. (c) Evaluate the main sum of Riemann–Siegel at many equally spaced points at once, and measure the speed-up.

## Solutions

### P14.1

In [ ]:
def Xi(t):
    s = mp.mpf(1)/2 + 1j*t
    return mp.re(s*(s-1)/2 * mp.pi**(-s/2) * mp.gamma(s/2) * mp.zeta(s))
omega = lambda y: mp.nsum(lambda n: mp.e**(-mp.pi*n*n*y), [1, mp.inf])
for x in [0, 0.5, 1]:
    left = mp.quad(lambda t: Xi(t)/(t*t+0.25)*mp.cos(x*t), [0, 10, 20, 40, 80])
    right = mp.pi/2*(mp.e**(-x/2) - 2*mp.e**(x/2)*omega(mp.e**(2*x)))
    print(f"x = {x}:  integral = {mp.nstr(left, 15)},  closed form = {mp.nstr(right, 15)}")

In [ ]:
def theta_near_i(delta):
    # Lemma 14.2: theta(i+delta) = delta^(-1/2) sum_m exp(-pi (m-1/2)^2 / delta)
    return delta**-0.5 * mp.nsum(lambda m: mp.e**(-mp.pi*(m-0.5)**2/delta), [-mp.inf, mp.inf])
def Phi(alpha):
    d = mp.e**(2j*alpha) - 1j
    return mp.re(mp.pi/2*(2*mp.cos(alpha/2) - mp.e**(1j*alpha/2)*theta_near_i(d)))
print("limit pi cos(pi/8) =", mp.nstr(mp.pi*mp.cos(mp.pi/8), 12))
for eps in [0.3, 0.1, 0.03, 0.01, 0.003]:
    a = mp.pi/4 - eps
    print(f"alpha = pi/4 - {eps}:  Phi = {mp.nstr(Phi(a), 12)},  |theta(e^(2i alpha))| = {mp.nstr(abs(theta_near_i(mp.e**(2j*a)-1j)), 3)}")
# direct check of the closed form against the integral at alpha = 0.5
a = 0.5
print("integral at alpha=0.5:", mp.nstr(mp.quad(lambda t: Xi(t)/(t*t+0.25)*mp.cosh(a*t), [0, 20, 40, 80, 160]), 12), " closed form:", mp.nstr(Phi(a), 12))

In [ ]:
for t in [10, 50, 100, 200, 400]:
    v = Xi(t)/(t*t+0.25)*mp.cosh(mp.pi*t/4)
    print(f"t = {t:4d}:  Xi(t) cosh(pi t/4)/(t^2+1/4) = {mp.nstr(v, 5):>10s}   |zeta(1/2+it)| t^(-1/4) = {mp.nstr(abs(mp.zeta(0.5+1j*t))*t**-0.25, 5)}")

The closed form matches the integral to all digits shown, and $\Phi(\alpha)$ converges to $\pi\cos\frac\pi8=2.9025\ldots$ as $\alpha\to\frac\pi4$, while $\theta(e^{2i\alpha})$ collapses faster than any power: at distance $0.01$ from $\frac\pi4$ it is about $10^{-16}$, at $0.003$ about $10^{-56}$. (The convergence of $\Phi$ itself is slow, with an overshoot near $0.03$: the correction is the term in $\cos\frac\alpha2$, whose distance to its limit decays only linearly.)

Yet the integrand at $\alpha=\frac\pi4$ does not decay at all: $\frac{\Xi(t)}{t^2+1/4}\cosh\frac{\pi t}4$ has the size of $t^{-1/4}|\zeta(\frac12+it)|$ times a constant, as the last table shows: the two columns are proportional, with ratio $\frac{2^{1/4}\sqrt{2\pi}}{4\pi^{1/4}}=0.5598$ in absolute value, by Stirling's formula and Exercise 14.1. So $\int_0^\infty$ does not converge at $\alpha=\frac\pi4$; the limit of $\Phi(\alpha)$ exists only thanks to the sign changes of $\Xi$, which make the large values cancel. This is exactly the tension that Hardy's proof exploits: if $\Xi$ kept one sign, monotone convergence would force the integral at $\frac\pi4$ to equal the finite limit, which is impossible.

### P14.2

In [ ]:
for N in [50, 100, 200, 400, 800, 1600]:
    m = np.arange(N)
    D = m[:, None] - m[None, :]
    H = np.where(D != 0, 1.0/np.where(D == 0, 1, D), 0.0)    # antisymmetric: eigenvalues are i*lambda
    lam = np.abs(np.linalg.eigvals(H)).max() if N <= 400 else np.linalg.norm(H, 2)
    print(f"N = {N:5d}:  largest |eigenvalue| = {lam:.5f},  pi - it = {math.pi - lam:.5f}")

In [ ]:
def meansq_poly(N, T):
    n = np.arange(1, N+1)
    t = np.linspace(0, T, int(T*20)+1)
    S = np.zeros(len(t), complex)
    for k in n:
        S += k**-0.5 * np.exp(-1j*t*math.log(k))
    integral = (getattr(np, "trapezoid", None) or np.trapz)(np.abs(S)**2, t)
    return integral - T*np.sum(1.0/n)
for N, T in [(10, 100), (50, 100), (100, 1000), (300, 300), (500, 2000)]:
    d = meansq_poly(N, T)
    print(f"N = {N:4d}, T = {T:5d}:  difference = {d:9.3f},  difference/N = {d/N:7.3f}   (Proposition 14.8: at most 16)")

The largest eigenvalue of the Hilbert matrix creeps up towards $\pi$ very slowly (the defect roughly halves each time $N$ doubles), so Schur's constant is sharp, as Exercise 14.6 shows with explicit vectors. The off-diagonal terms of the mean value are much smaller than the bound $16N$ of Proposition 14.8: the antisymmetry that kills the "$-1$" part does most of the work.

### P14.3

In [ ]:
rng = np.random.default_rng(14)
worst = 0
for t in rng.uniform(1, 3000, 25):
    x = t + 3; s = mp.mpf(1)/2 + 1j*t
    n = np.arange(1, int(x)+1)
    partial = np.sum(n**-0.5 * np.exp(-1j*t*np.log(n)))
    E = complex(mp.zeta(s)) - partial - complex(mp.mpf(x)**(1-s)/(s-1))
    worst = max(worst, abs(E)*math.sqrt(x))
print("max |E| sqrt(x) over 25 random t:", round(worst, 3), " (Lemma 14.6 allows 4)")

In [ ]:
h = 0.005
t1 = np.arange(0, 20, h); z1 = np.array([float(mp.siegelz(t)) for t in t1])
t2 = np.arange(20, 2000+h, h); z2 = Z_rs(t2)
tt = np.concatenate([t1, t2]); zz = np.concatenate([z1, z2])**2
cum = np.concatenate([[0], np.cumsum((zz[1:]+zz[:-1])/2*h)])
print("Ingham's constant 2*gamma - 1 =", round(2*0.5772156649 - 1, 4))
for T in [100, 250, 500, 1000, 1500, 2000]:
    i = np.searchsorted(tt, T-1e-9)
    print(f"T = {T:5d}:  (1/T) int |zeta|^2 - log(T/2pi) = {cum[i]/T - math.log(T/(2*math.pi)):.4f}")

The error in Lemma 14.6 is in practice about ten times smaller than the bound $4x^{-1/2}$. The mean square settles near Ingham's $2\gamma-1=0.1544$, with fluctuations of size about $T^{-1/2}\log T$, the size of Ingham's error term (Figure 14.1). Note that we computed $|\zeta|^2$ as $Z^2$: on the critical line $|\zeta(\frac12+it)|=|Z(t)|$ (Exercise 14.1).

### P14.4

In [ ]:
t0 = time.time()
g = zeros_rs(10, 10000, 0.01)
low = np.array([float(mp.zetazero(k).imag) for k in range(1, 151)])     # Riemann-Siegel is less accurate low down
g = np.concatenate([low, g[g > low[-1] + 0.5]])
print(len(g), "zeros up to 10^4 in", round(time.time()-t0, 1), "s")

N = 10**7
sieve = np.ones(N+1, bool); sieve[:2] = False
for p in range(2, int(N**0.5)+1):
    if sieve[p]: sieve[p*p::p] = False
lam = np.zeros(N+1)
for p in np.nonzero(sieve)[0]:
    q = p
    while q <= N:
        lam[q] = math.log(p); q *= p
n = np.arange(N+1.0); n[0] = 1
for x, t in [(10.0, 100.0), (50.0, 30.0), (200.0, 500.0)]:
    L = 2*np.sum(x**(1j*g)/(1+(t-g)**2) + x**(-1j*g)/(1+(t+g)**2))     # zeros come in conjugate pairs
    w = np.where(n <= x, (x/n)**(-0.5+1j*t), (x/n)**(1.5+1j*t))
    s = 0.5 + 1j*t
    dz = complex(mp.zeta(-0.5+1j*t, derivative=1)/mp.zeta(-0.5+1j*t))
    trivial = sum(x**(-2*k-0.5)/((1-2*k-s)*(1+s+2*k)) for k in range(1, 50))
    R = -x**-0.5*np.sum(lam*w) + 2*x**0.5/((2-s)*s) - x**(-1+1j*t)*dz - 2*trivial
    print(f"x = {x:5.0f}, t = {t:5.0f}:  zeros side = {L:.4f},  primes side = {R:.4f}")

In [ ]:
T = 5000.0; G = g[g <= T]; NT = len(G); ell = math.log(T/(2*math.pi))
u = G/(2*np.pi)*np.log(G/(2*np.pi*np.e))                 # unfolded zeros, mean spacing 1
D = u[:, None] - u[None, :]; W = 4/(4 + (2*np.pi*D/ell)**2)
print(NT, "zeros up to", T)
for a in [0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 1.0, 1.2, 1.5, 2.0]:
    F = np.sum(np.cos(2*np.pi*a*D)*W)/NT
    pred = min(a, 1) + ell*math.exp(-2*a*ell)
    print(f"alpha = {a:3.1f}:  F = {F:6.3f},  prediction = {pred:6.3f}")

Both sides of Montgomery's explicit formula agree to about $10^{-4}$, the size of the truncations (zeros above $10^4$, primes above $10^7$). For $F$, the agreement with $\alpha+\ell e^{-2\alpha\ell}$ is within a few hundredths for $\alpha<1$, and beyond $\alpha=1$, where nothing is proved, the computed values hover around $1$, as Montgomery conjectured (Figure 14.2). Without unfolding, that is with the normalisation $\frac{T\log T}{2\pi}$ of the theorem, the values would be about a third too small at this height, since $\log\frac T{2\pi e}/\log T=0.67$ for $T=5000$: asymptotic statements about zeros converge at the speed of $\log T$.

### P14.5

In [ ]:
def zeta_length_t(t):
    # Lemma 14.6 with x = t + 3: about t terms
    x = t + 3; n = np.arange(1, int(x)+1); s = 0.5 + 1j*t
    return np.sum(n**-0.5*np.exp(-1j*t*np.log(n))) + x**(1-s)/(s-1)
for t in [1000.0, 10000.0, 100000.0]:
    th = float(mp.siegeltheta(t))
    t0 = time.perf_counter(); a = abs(zeta_length_t(t)); ta = time.perf_counter()-t0
    t0 = time.perf_counter(); b = abs(Z_rs(t)[0]); tb = time.perf_counter()-t0
    t0 = time.perf_counter(); c = abs(complex(mp.zeta(0.5+1j*t))); tc = time.perf_counter()-t0
    print(f"t = {t:8.0f}: length-t sum {a:.6f} ({int(t+3)} terms, {ta*1e3:6.2f} ms)  "
          f"Riemann-Siegel {b:.6f} ({int(math.sqrt(t/2/math.pi))} terms, {tb*1e3:5.2f} ms)  mpmath {c:.6f} ({tc*1e3:6.1f} ms)")

In [ ]:
T = 10000.0
Nsmooth = float(mp.siegeltheta(T))/math.pi + 1
print("zeros found up to 10^4:", np.sum(g <= T), "   theta(T)/pi + 1 =", round(Nsmooth, 3))
gram = []   # Gram points: theta(g_n) = n pi
for k in range(0, 200):
    gram.append(float(mp.findroot(lambda t: mp.siegeltheta(t) - k*mp.pi, 2*math.pi*math.exp(1+mp.lambertw((8*k+1)/(8*math.e)).real))))
gram = np.array(gram)
zs = g[g < gram[-1]]
law = np.mean([np.sum((zs > gram[k]) & (zs <= gram[k+1])) == 1 for k in range(len(gram)-1)])
print(f"Gram's law (exactly one zero between consecutive Gram points) holds for {100*law:.1f}% of the first {len(gram)-1} Gram intervals")

In [ ]:
# Multi-evaluation: the main sum sum_{n<=N} n^(-1/2) exp(i(theta(t) - t log n)) on a grid t_j = t0 + j*delta.
# Each term rotates by the fixed factor n^(-i delta) from one grid point to the next: one complex
# multiplication per term instead of a logarithm, a cosine and a multiplication.
t0_, delta, M = 100000.0, 0.01, 4000
Nn = int(math.sqrt((t0_ + M*delta)/(2*math.pi)))
n = np.arange(1, Nn+1)
t = t0_ + delta*np.arange(M)
tic = time.perf_counter()
naive = np.array([np.sum(n**-0.5*np.cos(theta_rs(tj) - tj*np.log(n))) for tj in t])
t_naive = time.perf_counter() - tic
tic = time.perf_counter()
term = n**-0.5*np.exp(-1j*t0_*np.log(n)); step = np.exp(-1j*delta*np.log(n))
fast = np.empty(M)
for j in range(M):
    fast[j] = np.real(np.exp(1j*theta_rs(t[j]))*np.sum(term)); term *= step
t_fast = time.perf_counter() - tic
print(f"{M} values near t = 1e5 with {Nn} terms each: naive {t_naive:.2f} s, rotation {t_fast:.2f} s, max difference {np.max(np.abs(naive-fast)):.1e}")

(a) The sum of length $t$ of Lemma 14.6 is correct but needs $t$ terms; Riemann–Siegel needs only $\sqrt{t/2\pi}$, about $126$ at $t=10^5$, and is accurate to about $t^{-3/4}$ with one correction term. That is the whole difference between a proof tool and a computing tool.

(b) The $10\,142$ zeros found below $10^4$ agree with $\frac{\vartheta(T)}\pi+1=10\,142.97$; the difference is $S(T)$, which is small and has mean $0$ (Volume 5, Chapter 6). Turing's method makes this count rigorous. Gram's law, the observation that each interval between consecutive Gram points (where $\vartheta(g_n)=n\pi$) contains exactly one zero, holds for most intervals low down but fails more and more often higher up: it is a heuristic, not a theorem, and its failures are where $S(t)$ is large.

(c) When many values on a grid are wanted, each term $n^{-it}$ is updated by a fixed rotation, which removes the logarithms and cosines. Odlyzko and Schönhage (1988) went much further: they approximate the whole sum $\sum_nn^{-1/2-it}$ on a grid of $M$ points by a rational function evaluated with the fast Fourier transform and fast multipole-like expansions, at total cost about $(N+M)\,t^\varepsilon$ instead of $NM$. Odlyzko used it to compute hundreds of millions of zeros near the $10^{20}$-th and $10^{22}$-nd, and his histograms are the pictures of Volume 11, Chapter 11. The same idea (evaluate a sum of $N$ oscillating terms at $M$ points in nearly linear time) is the nonuniform fast Fourier transform used in medical imaging and radio astronomy.